# Extract ESM-2 Embeddings

This notebook extracts ESM-2 embeddings for CAFA 6 proteins and writes resumable `.npy` shard files under `artifacts/embeddings/`.

Run cells from top to bottom. Do not skip the manifest rebuild cell: it rewrites batch paths so they point to Google Drive paths inside Colab instead of Windows paths from your PC.

Recommended runtime workflow:

1. Start with **T4 GPU** and set `MAX_BATCHES = 1` in the extraction cell. This is a cheap smoke test for installs, paths, batch loading, and shard writing.
2. After one train shard writes successfully, switch to **L4 GPU** for the full extraction.
3. Run full train extraction with `SPLIT = 'train'` and `MAX_BATCHES = None`.
4. Run full test extraction with `SPLIT = 'test'` and `MAX_BATCHES = None`.

Do not use TPU for this notebook. The code uses `torch` and `fair-esm` on CUDA GPUs.

If you hit CUDA out-of-memory, use `INFERENCE_BATCH_SIZE = 1` and keep `MAX_SEQUENCE_LENGTH = 1022`. Very long proteins are truncated for embedding extraction because ESM-2 attention memory grows roughly with sequence length squared.

## Cell 1: Connect The Notebook To Your Project

This cell sets `PROJECT_ROOT`, changes Colab's working directory to the project, and imports the extraction helpers from `cafa6.embeddings`.

Before running it, confirm that `PROJECT_ROOT` matches where you uploaded the project in Google Drive. The expected layout is:

```text
/content/drive/MyDrive/<your-project-folder>/src
  cafa6/
  scripts/
  data/processed/
  artifacts/embeddings/
  notebooks/
```

If your folder is somewhere else, edit only the `PROJECT_ROOT = ...` line.

In [ ]:
# Change this if your uploaded project is not in path.
# This must be the folder that contains cafa6/, scripts/, data/, artifacts/, and notebooks/.
import os
import sys
from pathlib import Path
w
PROJECT_ROOT = Path('Path')

# Make cafa6 importable and make all relative paths resolve from the project root.
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Import reusable package code. The notebook stays thin; core extraction logic lives in cafa6/embeddings.py.
from cafa6.embeddings import (
    DEFAULT_MODEL_NAME,
    extract_manifest_batches,
    manifest_summary,
    pending_batch_ids,
    read_manifest,
)

import inspect
signature = inspect.signature(extract_manifest_batches)
assert 'max_sequence_length' in signature.parameters, 'Your Colab copy of cafa6/embeddings.py is stale. Re-sync the updated file.'

print(f'Project root: {PROJECT_ROOT}')
print(f'Current working directory: {Path.cwd()}')
print(f'extract_manifest_batches signature: {signature}')

## Cell 2: Rebuild Manifests Inside Colab

This cell prepares the embedding batch files and manifests using Colab paths.

It reads:

```text
data/processed/train_sequences.parquet
data/processed/test_sequences.parquet
```

It writes or refreshes:

```text
artifacts/embeddings/esm2_train/manifest.csv
artifacts/embeddings/esm2_train/batches/batch_*.parquet
artifacts/embeddings/esm2_test/manifest.csv
artifacts/embeddings/esm2_test/batches/batch_*.parquet
```

This step is safe to rerun. It does not run ESM-2. It only prepares batch metadata and batch parquet files.

## Cell 3: Choose Train Or Test Split

Use this cell to choose which manifest to read.

First run:

```python
SPLIT = 'train'
```

After train extraction finishes, change it to:

```python
SPLIT = 'test'
```

This cell does not extract embeddings. It only loads the selected manifest and prints a summary.

In [ ]:
# Run train first. After train is fully complete, change this to 'test' and rerun this cell plus the cells below.
SPLIT = 'train'  # change to 'test' after train completes

# This manifest lists every protein sequence, the batch file containing it, and the shard file that should be written.
MANIFEST_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / f'esm2_{SPLIT}' / 'manifest.csv'

# fair-esm model loader name. This 650M model is a strong default, but it is large.
MODEL_NAME = DEFAULT_MODEL_NAME

# ESM-2 layer to pool. For esm2_t33_650M_UR50D, layer 33 is the final representation layer.
REPR_LAYER = 33

# Truncate very long proteins before ESM-2 tokenization.
# This prevents single long proteins from causing CUDA OOM.
# 1022 is the practical ESM-2 residue limit used for stable extraction.
MAX_SEQUENCE_LENGTH = 1022

# Number of sequences sent through the GPU at once inside a shard.
# If you hit CUDA out-of-memory, lower this to 4, 2, or 1.
INFERENCE_BATCH_SIZE = 1

manifest = read_manifest(MANIFEST_PATH)
manifest_summary(manifest)

## Cell 4: Check What Still Needs To Run

This cell checks the manifest and returns batch IDs whose shard files are still missing.

If Colab disconnects, rerun from the top. Existing shard files are detected, and completed batches are skipped.

In [ ]:
# pending is a list like ['batch_00000', 'batch_00001', ...].
# A batch is considered complete when its shard_*.npy file exists.
pending = pending_batch_ids(manifest)
print(f'Pending batches: {len(pending)}')
pending[:10]

## Cell 5: Extract Embeddings

This is the expensive GPU cell.

For the first run on T4, set:

```python
MAX_BATCHES = 1
```

That should write one file such as:

```text
artifacts/embeddings/esm2_train/shard_00000.npy
```

After that one-batch test works, switch to L4 GPU and set:

```python
MAX_BATCHES = None
```

Then run full train. After train is done, change `SPLIT` to `'test'` in Cell 3, rerun Cells 3-5, and extract full test.

### GPU Check

Run this before the extraction cell. It confirms whether PyTorch sees the GPU. Colab may still show a low-utilization warning during setup cells because those cells only prepare files and do not use the GPU.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
    print('Reserved GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))

!nvidia-smi

In [ ]:
# First smoke test: set MAX_BATCHES = 1 on T4 GPU.
# Full extraction: set MAX_BATCHES = None, preferably on L4 GPU.
MAX_BATCHES = None  # set to an integer for short Colab sessions

# This loads the ESM-2 model once, then processes pending batches in order.
# Each completed batch writes one shard_*.npy file and updates manifest.csv.
written = extract_manifest_batches(
    manifest_path=MANIFEST_PATH,
    batch_ids=pending,
    model_name=MODEL_NAME,
    repr_layer=REPR_LAYER,
    inference_batch_size=INFERENCE_BATCH_SIZE,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_batches=MAX_BATCHES,
)

# Print shard paths written during this cell execution.
for shard_path in written:
    print(shard_path)

# Reload the manifest so the summary reflects completed shards.
manifest = read_manifest(MANIFEST_PATH)
manifest_summary(manifest)